In [1]:
# !pip install backbones

In [8]:
from torch.utils.data import Dataset
import os, random
import cv2
import numpy as np
from torchvision import transforms

class TripletFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = '/projectnb/cs585bp/students/dlgirija/gold_ivc/IVC_Project/output_unmask'
        self.pos_dir = '/projectnb/cs585bp/students/dlgirija/gold_ivc/IVC_Project/output_texture'
        self.classes = [f for f in os.listdir(self.root_dir) if not f.startswith('.')]
        transform = transforms.Compose([
            transforms.ToTensor(),  # Converts to [0, 1] float tensor
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # To [-1, 1]
        ])

        self.transform = transform

    def __len__(self):
        return 100000  # Or dynamically count combinations

    def __getitem__(self, idx):

        anchor_class = random.choice(self.classes)
        positive_class = anchor_class
        while len(os.listdir(os.path.join(self.pos_dir, anchor_class))) == 0 or len(os.listdir(os.path.join(self.root_dir, anchor_class))) == 0 or anchor_class=='.ipynb_checkpoints':
            anchor_class = random.choice([c for c in self.classes if c != anchor_class])
            positive_class = anchor_class
        
        negative_class = random.choice([c for c in self.classes if c != anchor_class])
        while len(os.listdir(os.path.join(self.pos_dir, negative_class))) == 0 or negative_class=='.ipynb_checkpoints':
            negative_class = random.choice([c for c in self.classes if c != anchor_class])
            
        anchor_img = random.choice(os.listdir(os.path.join(self.root_dir, anchor_class)))
        while anchor_img == '.ipynb_checkpoints':
            anchor_img = random.choice(os.listdir(os.path.join(self.root_dir, anchor_class)))
        
        positive_img = random.choice(os.listdir(os.path.join(self.pos_dir, positive_class)))
        while positive_img == '.ipynb_checkpoints':
            positive_img = random.choice(os.listdir(os.path.join(self.pos_dir, positive_class)))
        
        negative_img = random.choice(os.listdir(os.path.join(self.pos_dir, negative_class)))
        while negative_img == '.ipynb_checkpoints':
            negative_img = random.choice(os.listdir(os.path.join(self.pos_dir, negative_class)))
        
        anchor = cv2.imread(os.path.join(self.root_dir, anchor_class, anchor_img.replace('.ppm','.jpg')))
        positive = cv2.imread(os.path.join(self.pos_dir, positive_class, positive_img.replace('.ppm','.jpg')))
        negative = cv2.imread(os.path.join(self.pos_dir, negative_class, negative_img.replace('.ppm','.jpg')))
        # print(os.path.join(self.pos_dir, negative_class, negative_img))
        try:
            anchor = cv2.resize(anchor, (112, 112))
            positive = cv2.resize(positive, (112, 112))
            negative = cv2.resize(negative, (112, 112))
        except:
            print(os.path.join(self.pos_dir, negative_class, negative_img),anchor_img, positive_img)
            

        if self.transform:
            anchor = cv2.cvtColor(anchor, cv2.COLOR_BGR2RGB)
            positive = cv2.cvtColor(positive, cv2.COLOR_BGR2RGB)
            negative = cv2.cvtColor(negative, cv2.COLOR_BGR2RGB)
        
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)

        return anchor, positive, negative


In [9]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [10]:
import sys
sys.path.append('/projectnb/cs585bp/students/dlgirija/gold_ivc/IVC_Project/insightface/recognition/arcface_torch')

In [11]:
# from backbones import get_model
# import torch.nn.functional as F

# model = get_model("r100", fp16=False)
# # model.eval()


In [22]:
from torch.utils.data import DataLoader

# Define your dataset directory and transformations
dataset_dir = ''

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Create dataset and loader
train_dataset = TripletFaceDataset(root_dir=dataset_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8)


In [24]:
import torch
import torch.nn as nn
from torchvision.models import resnet101

class CustomResNet18(nn.Module):
    def __init__(self):
        super(CustomResNet18, self).__init__()
        model = resnet101(pretrained=True)
        self.features = nn.Sequential(*list(model.children())[:-1])  # Remove the last layer (fc)
        self.fc = nn.Linear(model.fc.in_features, 256)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # Flatten all dimensions except batch
        x = self.fc(x)
        return x

# Example usage:
# num_classes = 128  # Example number of output classes (for embeddings or classification)
model = CustomResNet18()

# Print the modified model architecture
# print(model)


/share/pkg.8/academic-ml/fall-2024/install/fall-2024-pyt/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /usr3/graduate/dlgirija/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100%|██████████| 171M/171M [00:00<00:00, 432MB/s] 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision.models import resnet18

class TripletNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        # return self

    def forward(self, anchor, positive, negative):
        anchor[anchor == -1] = 0
        positive[positive == -1] = 0
        negative[negative == -1] = 0
        a = F.normalize(self.backbone(anchor), p=2, dim=1)
        p = F.normalize(self.backbone(positive), p=2, dim=1)
        n = F.normalize(self.backbone(negative), p=2, dim=1)
        # print(self.backbone(anchor))
        # print("Anchor min/max:", a.min(), anchor.max())
        # print("Anchor has NaNs?", torch.isnan(a).any())
        return a, p, n


# model = resnet18(pretrained=True)
new_model = TripletNet(model)
new_model = new_model.cuda()
# new_model.eval()
optimizer = optim.Adam(new_model.parameters(), lr=1e-4)
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=1)
num_epochs = 100
torch.cuda.empty_cache()
for epoch in range(num_epochs):
    for anchor, positive, negative in train_loader:
        anchor, positive, negative = anchor.cuda(), positive.cuda(), negative.cuda()
        out_a, out_p, out_n = new_model(anchor, positive, negative)

        loss = triplet_loss(out_a, out_p, out_n)
        # print(loss.min(), out_a)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch% 10 == 0:
            torch.save(new_model.state_dict(), 'triplet_finetuned_r101_1_'+str(epoch)+'_loss_000_min.pth')
        # torch.cuda.empty_cache()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.1590
Epoch 1, Loss: 0.0364
Epoch 2, Loss: 0.0128
Epoch 3, Loss: 0.0944
Epoch 4, Loss: 0.0111
Epoch 5, Loss: 0.0260
Epoch 6, Loss: 0.0000
Epoch 7, Loss: 0.0000
Epoch 8, Loss: 0.0152
Epoch 9, Loss: 0.0000
Epoch 10, Loss: 0.0000
Epoch 11, Loss: 0.0000
Epoch 12, Loss: 0.0000
Epoch 13, Loss: 0.0000
Epoch 14, Loss: 0.0292
Epoch 15, Loss: 0.2288
Epoch 16, Loss: 0.0000
Epoch 17, Loss: 0.0000
Epoch 18, Loss: 0.0000
Epoch 19, Loss: 0.0000
Epoch 20, Loss: 0.1022
Epoch 21, Loss: 0.0000
Epoch 22, Loss: 0.0000
Epoch 23, Loss: 0.0000
Epoch 24, Loss: 0.1330
Epoch 25, Loss: 0.0000
Epoch 26, Loss: 0.0000
Epoch 27, Loss: 0.0000
Epoch 28, Loss: 0.0000
Epoch 29, Loss: 0.0000
Epoch 30, Loss: 0.0000
Epoch 31, Loss: 0.0577
Epoch 32, Loss: 0.0000
Epoch 33, Loss: 0.0000
Epoch 34, Loss: 0.0000
Epoch 35, Loss: 0.0251
Epoch 36, Loss: 0.0000
Epoch 37, Loss: 0.0000
Epoch 38, Loss: 0.0000
Epoch 39, Loss: 0.0000


In [20]:
# torch.save(new_model.state_dict(), 'triplet_finetuned_r18_2_10_loss_004.pth')

In [ ]:
git clone -b pytorch https://github.com/deepinsight/insightface.git